# Fire research: SwinUNETR on Google Colab

Colab copy of the Kaggle notebook (same settings, same trainer). Select a **GPU runtime**:
**Runtime → Change runtime type → T4 GPU** (or L4/A100). In the **Secrets** panel (key icon in the left
sidebar), add `WANDB` (your W&B API key) and `HF_TOKEN` (your Hugging Face read token) and switch on
**Notebook access** for both. `HF_TOKEN` is optional for a public dataset or data already in Google Drive.

Run the cells in order. Loss and learning rate follow the TS-SatFire paper's prediction setup
(`LOSS = 'dicece'`: 1:1 Dice + cross entropy, class weights 1.0 / 446.7836; `LEARNING_RATE = 0.001`).
Other defaults follow the paper's SwinUNETR-3D prediction setup: batch size 8,
`TS_LENGTH = 6` input days, feature size 36, and 100 epochs (3 attention heads; the paper gives no head count).
**New, not in the paper:** `USE_DICE_FOCAL = True` trains with Dice + focal loss (`--loss dicefocal`,
weights `LAMBDA_DICE` / `LAMBDA_FOCAL`) instead of `LOSS`. `LR_SCHEDULE` optionally decays the learning rate
(`'plateau'` or `'cosine'`); the default `'none'` keeps the paper's constant rate.
Set `EPOCHS = 1` for an initial end-to-end check. No GPU memory capacity is assumed.

The training code is cloned from GitHub (`REPO_BRANCH`, optionally pinned to `REPO_COMMIT`) and run as
`python -m firepred.cli.train`; this notebook only holds settings and calls
`firepred.data.windows_master` (data) and `firepred.training.config.RunConfig` (the run). `MODEL_NAME` picks any model registered in
`firepred/models/zoo/` (`MODEL_OPTIONS` passes its extra settings as `--model-option KEY=VALUE`).
`USE_TESSERA` switches the experiment:
- `False`: 27 stored channels, 43 model channels; downloads only `data.zip`.
- `True`: also downloads `tessera.zip` from `SamuelWu318/ts-tesserafire`, drops the fires
  without prior-year TESSERA (all 2017 fires), and trains with 172 model channels.
  To train the matched no-TESSERA baseline on the same fires, keep `USE_TESSERA = True`
  and set `TESSERA_INPUTS = False`.

**Storage.** Code and data go under `/content/fire-research` on the Colab VM, which is wiped when the
runtime disconnects. The window `data.zip` is about 22 GB and is extracted beside it, so a standard
runtime's disk is enough for one data set. W&B logs and checkpoints go to `OUTPUT_DIR`: the VM by default,
or Google Drive with `SAVE_TO_DRIVE = True` (mounted in the settings cell) so they survive a disconnect.
Each best checkpoint is also uploaded as a versioned W&B **model Artifact** under `gt-fri/AFBAPred`,
so a finished run's checkpoints are safe either way.

**Session limits.** Colab may disconnect idle or long sessions. A 100-epoch run can outlast a free
session; this trainer does not resume, so start with fewer `EPOCHS` or a longer-lived runtime.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

ACCELERATOR = 'gpu'     # Colab: one GPU.
DEVICE = 'cuda'

# Experiment settings: every one maps to a flag of `python -m firepred.cli.train`.
MODEL_NAME = 'swinunetr3d'  # Any registered model: swinunetr3d, swinunetr3d_tpool, unet3d, attunet, ...
MODEL_OPTIONS = {}         # Model-specific settings, e.g. {'width': 32}; passed as --model-option KEY=VALUE.
BATCH_SIZE = 8          # Paper's SwinUNETR-3D prediction batch size; reduce if CUDA runs out of memory.
EPOCHS = 100            # Set to 1 for an initial training/validation check.
LEARNING_RATE = 0.001   # Adam learning rate; the paper's prediction setting is 0.001.
USE_DICE_FOCAL = False  # NEW, not in the paper: True trains with Dice + focal ('dicefocal') instead of LOSS.
LOSS = 'dicece'         # Used when USE_DICE_FOCAL is False. 'dicece': paper's 1:1 Dice + CE;
                        # 'dice': released-code Dice loss.
# Dice + CE settings (used only with LOSS = 'dicece'; LAMBDA_DICE also with USE_DICE_FOCAL); defaults are the paper's.
# Class weights (background, fire); MONAI applies them to Dice and CE. The paper's is (1.0, 446.7836);
# this notebook has trained with (1.0, 457.1) (docs/KNOWN_ISSUES.md #12).
CE_WEIGHT = (1.0, 457.1)
LAMBDA_DICE = 1.0       # Weight of the Dice term.
LAMBDA_CE = 1.0         # Weight of the cross-entropy term (1:1 with Dice in the paper).
# Focal settings (used only with USE_DICE_FOCAL = True). Unweighted Dice starts near 0.7 and focal near 0.1.
LAMBDA_FOCAL = 1.0      # Weight of the focal term.
FOCAL_GAMMA = 2.0       # Focusing exponent; 0 = plain binary cross entropy.
FOCAL_ALPHA = None      # Weight of fire pixels in the focal term, e.g. 0.25; None = off.
if USE_DICE_FOCAL:
    LOSS = 'dicefocal'  # LAMBDA_DICE x Dice + LAMBDA_FOCAL x focal loss on the fire channel, no CE.
# NEW, NOT IN THE TS-SATFIRE PAPER OR THE OFFICIAL CODE: optional learning-rate decay.
# 'none' keeps the paper's constant LEARNING_RATE. 'plateau' multiplies it by LR_DECAY_FACTOR
# after LR_PATIENCE epochs without a lower validation loss; 'cosine' decays it smoothly to
# LR_MIN over EPOCHS. Runs with decay get '_lr-<schedule>' in their W&B name and checkpoints.
LR_SCHEDULE = 'none'    # NEW: 'none' (paper) | 'plateau' | 'cosine'
LR_DECAY_FACTOR = 0.5   # NEW, plateau only: multiply the learning rate by this.
LR_PATIENCE = 5         # NEW, plateau only: epochs without improvement before decaying.
LR_MIN = 1e-6           # NEW, plateau and cosine: the lowest learning rate.
# NEW, NOT IN THE TS-SATFIRE PAPER: score the test fires with the current weights every
# TEST_EVERY epochs and chart test F1/IoU. Logging only: no gradients, eval mode, and the
# best checkpoint is still chosen by validation loss. 0 turns it off.
TEST_EVERY = 10         # NEW: epochs between test checks during training.
NUM_HEADS = 3           # Attention heads in each Swin stage.
FEATURE_SIZE = 36       # Paper's feature size (24 and 36 tested, 36 picked); must be divisible by NUM_HEADS.
NUM_WORKERS = 2         # Each worker builds whole batches in host RAM; keep low on standard Colab RAM.
UNZIP_DIR = 'off'       # 'off': read the window archives; 'auto': unzip once beside the data (~5.5x disk).
RUN_GPU_SMOKE_TEST = True  # Test one training step before the full experiment.

# Window settings; must match how the window data was built (python -m firepred.cli.dataset process -ts/-it).
# The TS-SatFire paper's prediction benchmark uses 6 input days.
TS_LENGTH = 6           # Input days per window.
TS_INTERVAL = 1         # Days between window starts.

# TESSERA switch. Only USE_TESSERA = True downloads or reads any TESSERA data.
USE_TESSERA = True     # True: drop 2017 fires and load per-fire TESSERA embeddings.
TESSERA_INPUTS = True   # With USE_TESSERA: False trains the matched baseline (same fires, 43 channels).

# Training code is cloned from GitHub; set REPO_COMMIT to pin an exact version.
REPO_URL = 'https://github.com/SamuelWu318/fire-research-and-innovation.git'
REPO_BRANCH = 'explain'  # Branch with the firepred package; 'main' once it is merged.
REPO_COMMIT = None      # Example: 'd252458'

WANDB_ENTITY = 'gt-fri' # Team with write permission for your account.
WANDB_PROJECT = 'AFBAPred'  # Already configured in the bundled trainer.
WANDB_SECRET = 'WANDB'  # Name of the Colab secret holding your W&B API key.
HF_SECRET = 'HF_TOKEN'  # Read token for private/gated Hugging Face data.
# None: the repo for TS_LENGTH / TS_INTERVAL (SamuelWu318/ts-satfire-processed-{TS_LENGTH}-{TS_INTERVAL}).
# Set a repo id to override.
HF_DATASET = None                                       # data.zip
TESSERA_DATASET = 'SamuelWu318/ts-tesserafire'           # tessera.zip (TESSERA mode only)

# Leave both unset to download data.zip from the Hugging Face dataset above.
# Alternatively, point ONE of these at a copy in Google Drive (mounted automatically).
# Reading a 22 GB zip through Drive is slow; an extracted folder or a fresh download is usually faster.
ATTACHED_DATA_ROOT = None  # Example: '/content/drive/MyDrive/fire-data/data'
ATTACHED_ZIP = None        # Example: '/content/drive/MyDrive/fire-data/data.zip'
# TESSERA mode only: leave unset to download tessera.zip, or set ONE of these.
ATTACHED_TESSERA_DIR = None  # Folder with p<id>.npz and excluded_npz.txt
ATTACHED_TESSERA_ZIP = None  # Example: '/content/drive/MyDrive/fire-data/tessera.zip'

# Colab storage: the VM disk is wiped on disconnect; Drive keeps outputs.
SAVE_TO_DRIVE = False      # True: W&B logs and checkpoints go to Google Drive (DRIVE_OUTPUT_DIR).
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/fire-research-output'

# Mount Google Drive when an output or input path lives there.
_drive_paths = [p for p in (ATTACHED_DATA_ROOT, ATTACHED_ZIP, ATTACHED_TESSERA_DIR, ATTACHED_TESSERA_ZIP)
                if p and str(p).startswith('/content/drive')]
if SAVE_TO_DRIVE or _drive_paths:
    from google.colab import drive
    drive.mount('/content/drive')

TEMP_ROOT = Path('/content/fire-research')
CODE_DIR = TEMP_ROOT / 'code'
# Data folders under TEMP_ROOT come from firepred.data.windows_master.workspace (section 4).
OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR) if SAVE_TO_DRIVE else TEMP_ROOT / 'output'
for folder in (TEMP_ROOT, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

assert BATCH_SIZE >= 1 and EPOCHS >= 1 and NUM_WORKERS >= 0
assert not (ATTACHED_DATA_ROOT and ATTACHED_ZIP), 'Choose a data folder OR a ZIP.'
assert not (ATTACHED_TESSERA_DIR and ATTACHED_TESSERA_ZIP), 'Choose a TESSERA folder OR a ZIP.'
print('model:', MODEL_NAME, MODEL_OPTIONS or '', '| USE_DICE_FOCAL:', USE_DICE_FOCAL, '| USE_TESSERA:', USE_TESSERA, '| TESSERA inputs:', USE_TESSERA and TESSERA_INPUTS,
      '| loss:', LOSS,
      *(('| CE weight:', CE_WEIGHT, '| lambda dice/ce:', LAMBDA_DICE, LAMBDA_CE) if LOSS == 'dicece' else ()),
      *(('| lambda dice/focal:', LAMBDA_DICE, LAMBDA_FOCAL, '| focal gamma/alpha:', FOCAL_GAMMA, FOCAL_ALPHA)
        if LOSS == 'dicefocal' else ()),
      '| lr:', LEARNING_RATE, '| lr schedule (NEW, not in paper):', LR_SCHEDULE, '| test every (NEW):', TEST_EVERY, '| code:', REPO_BRANCH,
      '| windows:', TS_LENGTH, 'days every', TS_INTERVAL,
      '| data:', HF_DATASET or 'repo for these windows', '| outputs:', OUTPUT_DIR)


## 1. Install the training dependencies
Keep Colab's installed PyTorch/CUDA/torchvision combination. Geospatial packages are not
required to train from already-processed NPZ files. MONAI 1.3.2 matches the supplied setup. TESSERA mode needs no extra packages.


In [ ]:
import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime (Runtime -> Change runtime type), then rerun from the top.')
TORCHVISION_VERSION, TORCH_XLA_VERSION = torchvision.__version__, None
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
for index in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {gpu.name}; {gpu.total_memory / 1024**3:.1f} GiB VRAM')

# Constraints prevent pip from replacing Colab's working GPU libraries.
constraints = TEMP_ROOT / 'torch-constraints.txt'
constraints.write_text(
    f'torch=={torch.__version__}\ntorchvision=={torchvision.__version__}\n'
)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--constraint', str(constraints),
    'monai==1.3.2', 'torchinfo==1.8.0', 'einops==0.8.0',
    'wandb', 'huggingface_hub', 'matplotlib', 'pandas', 'scikit-learn', 'tqdm',
], check=True)


## 2. Clone the training code
Clones `REPO_BRANCH` of the GitHub repository into the notebook's code folder (replacing
any earlier clone), checks the trainer's CLI (`python -m firepred.cli.train --help`) and makes the
`firepred` package importable here. The exact commit is printed and saved
beside the checkpoints. Nothing is pushed to GitHub.

In [ ]:
import shutil

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)  # Always start from a clean checkout.
subprocess.run(['git', 'clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, str(CODE_DIR)],
               check=True)
if REPO_COMMIT:
    subprocess.run(['git', '-C', str(CODE_DIR), 'checkout', '--quiet', REPO_COMMIT], check=True)
CODE_COMMIT = subprocess.run(['git', '-C', str(CODE_DIR), 'rev-parse', 'HEAD'],
                             capture_output=True, text=True, check=True).stdout.strip()
print(f'Training code: {REPO_URL} @ {REPO_BRANCH} {CODE_COMMIT}')

help_result = subprocess.run(
    [sys.executable, '-m', 'firepred.cli.train', '--help'],
    cwd=CODE_DIR, text=True, capture_output=True,
)
if help_result.returncode:
    print(help_result.stdout)
    print(help_result.stderr)
    help_result.check_returncode()
for flag in ('--epochs', '--num-workers', '--data-root', '--wandb-user',
             '--tessera', '--upload-checkpoints', '--loss', '--ce-weight',
             '--lambda-dice', '--lambda-ce', '--lr-schedule', '--test-every', '--model-option'):
    assert flag in help_result.stdout, f'Training script is missing {flag}; check REPO_BRANCH'
print('Training imports and command-line options passed.')

# Notebook helpers (data staging, window repo names) come from the cloned package.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

## 3. Authenticate W&B
This reads the `WANDB` Colab secret and creates (then deletes) a small `colab-permission-check` run in `gt-fri/AFBAPred`.
The training subprocess inherits the key through its environment.


In [ ]:
from google.colab import userdata
from firepred.training.tracker import check_wandb_access

os.environ['WANDB_API_KEY'] = userdata.get(WANDB_SECRET)  # raises if missing or not shared
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_DIR'] = str(OUTPUT_DIR)
os.environ['WANDB_CACHE_DIR'] = str(TEMP_ROOT / 'wandb-cache')
os.environ['WANDB_CONFIG_DIR'] = str(TEMP_ROOT / 'wandb-config')
os.environ['WANDB_DATA_DIR'] = str(TEMP_ROOT / 'wandb-staging')
# Logs one tiny run with this key and deletes it again (firepred/training/tracker.py).
print('Permission-check run:', check_wandb_access(WANDB_ENTITY, WANDB_PROJECT, name='colab-permission-check'))


## 4. Prepare data
Download from the Hugging Face dataset, or use the Google Drive path configured above. Both `train/val/test` and
`data/train`, `data/val`, `data/test` ZIP layouts are supported.
Extraction checks disk space and reuses a completed extraction within this session.
For large archives, an already-extracted folder in Drive avoids keeping both a ZIP and its
extracted contents on the VM disk (reading through Drive is slower).


With `USE_TESSERA = True`, `tessera.zip` is also downloaded (or read from the attached
TESSERA path) and a symlink view is built: every kept window archive plus `tessera/`.
Fires listed in `excluded_npz.txt` (no prior-year TESSERA) are left out of the view;
the extracted data itself is never modified, so both modes can reuse one extraction.
With `USE_TESSERA = False`, no TESSERA file is downloaded or read.

In [ ]:
# Data staging lives in firepred/data/windows_master.py; this cell only passes the settings.
from firepred.data import windows_master

work = windows_master.workspace(TEMP_ROOT, TS_LENGTH, TS_INTERVAL, use_tessera=USE_TESSERA, window_repo=HF_DATASET)
print('Window data repo:', work.window_repo)
staged = windows_master.stage_training_data(
    work,
    token=windows_master.read_secret(HF_SECRET),   # Kaggle secret / Colab secret / environment; None = public
    tessera_repo=TESSERA_DATASET,
    attached_data_root=ATTACHED_DATA_ROOT, attached_zip=ATTACHED_ZIP,
    attached_tessera_dir=ATTACHED_TESSERA_DIR, attached_tessera_zip=ATTACHED_TESSERA_ZIP,
)
DATA_ROOT, TRAIN_ROOT = staged.data_root, staged.train_root


## 5. Check shapes and GPU memory
`python -m firepred.cli.check data` inspects one training batch and one validation sample. With the smoke test enabled,
it also performs one disposable forward/backward/Adam step using the same model settings.
This temporary model is discarded; the full training run starts fresh.
It uses the same `USE_TESSERA` setting and `MODEL_CHANNELS` as training.
If CUDA runs out of memory, reduce `BATCH_SIZE` and rerun from the configuration cell.


In [ ]:
# Shape check and, with RUN_GPU_SMOKE_TEST, one disposable training step (python -m firepred.cli.check data).
# Uncomment to run before the full experiment.
# check = [sys.executable, '-u', '-m', 'firepred.cli.check', 'data',
#          '--data-root', str(TRAIN_ROOT), '--batch', str(BATCH_SIZE), '-ts', str(TS_LENGTH),
#          '-m', MODEL_NAME, '-nh', str(NUM_HEADS), '-ed', str(FEATURE_SIZE), '-lr', str(LEARNING_RATE),
#          '--loss', LOSS, '--ce-weight', str(CE_WEIGHT[0]), str(CE_WEIGHT[1]),
#          '--lambda-dice', str(LAMBDA_DICE), '--lambda-ce', str(LAMBDA_CE)]
# check += [arg for key, value in MODEL_OPTIONS.items() for arg in ('--model-option', f'{key}={value!r}')]
# if LOSS == 'dicefocal':
#     check += ['--lambda-dice', str(LAMBDA_DICE), '--lambda-focal', str(LAMBDA_FOCAL), '--focal-gamma', str(FOCAL_GAMMA)]
#     check += ['--focal-alpha', str(FOCAL_ALPHA)] if FOCAL_ALPHA is not None else []
# if USE_TESSERA and TESSERA_INPUTS:
#     check.append('--tessera')
# if RUN_GPU_SMOKE_TEST:
#     check.append('--train-step')
# subprocess.run(check, cwd=CODE_DIR, check=True)

### Check data time to load

In [ ]:
assert (CODE_DIR / 'firepred').is_dir(), 'Run the clone cell (section 2) first'
subprocess.run([sys.executable, '-m', 'firepred.cli.check', 'timing',
                '--data-root', str(TRAIN_ROOT), '-ts', str(TS_LENGTH)],
               cwd=CODE_DIR, check=True)

## 6. Train
The subprocess uses explicit code and data paths, so notebook directory changes do
not affect it. Logs stream below. Checkpoints remain in the W&B run's directory under
`OUTPUT_DIR/wandb` (Drive with `SAVE_TO_DRIVE = True`) even if training later fails.
Each launch starts fresh; this trainer does not implement training resume.
To find cloud checkpoints, open the training run's **Artifacts** tab. Upload failures
stop training with an error; the newly saved local checkpoint remains on disk.

W&B charts per epoch: `Training/Validation <loss> — Next-Day Fire Growth`; with
`LOSS = 'dicece'` also `Training/Validation Weighted Dice Loss` and `... Weighted
Cross-Entropy Loss`; `Training/Validation F1 — Next-Day Fire Growth` and `— Background`;
`Training/Validation IoU — Next-Day Fire Growth` (the paper's IoU, Equation 2; logged, never trained on);
`Validation Mean IoU` and `Validation Mean Dice (F1)` (these average in the background class, so they
are not the paper's numbers). **New, not in the paper:** with `TEST_EVERY > 0`, `Test F1/IoU — Next-Day
Fire Growth (periodic, current weights)` every `TEST_EVERY` epochs; logging only. Dice+CE totals are around 220 because
MONAI applies the 446.78 class weight to the Dice term too, so compare runs with different
losses by F1/IoU, not by loss value. After the last epoch the run reloads its best
checkpoint (lowest validation loss), scores the 24 test fires and logs `Test F1 — Next-Day Fire Growth`,
`Test IoU — Next-Day Fire Growth`, `Test F1 — Background`, `test_f1` and `test_iou` to the same run
(also in its summary). Pass `--no-test-after-train` to skip that step.

With TESSERA inputs, checkpoint names end in `_tessera.pth` and record the TESSERA
normalization; test such a checkpoint with `--tessera` and the same training root.
Dice+CE checkpoints end in `_dicece.pth`.

In [ ]:
# The run is one RunConfig (firepred/training/config.py): it checks the settings, derives the
# channel count and every name, and turns into the exact training command (argv_for).
from firepred.cli.train import argv_for, write_run_record
from firepred.training.config import LossConfig, RunConfig, ScheduleConfig

config = RunConfig(
    model=MODEL_NAME, mode='pred', batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    num_heads=NUM_HEADS, feature_size=FEATURE_SIZE, ts_length=TS_LENGTH, ts_interval=TS_INTERVAL,
    seed=42, epochs=EPOCHS, num_workers=NUM_WORKERS, device=DEVICE, data_root=TRAIN_ROOT,
    unzip_dir=UNZIP_DIR, tessera=USE_TESSERA and TESSERA_INPUTS,   # reads TRAIN_ROOT/tessera
    loss=LossConfig(LOSS, CE_WEIGHT, LAMBDA_DICE, LAMBDA_CE, LAMBDA_FOCAL, FOCAL_GAMMA, FOCAL_ALPHA),
    schedule=ScheduleConfig(LR_SCHEDULE, LR_DECAY_FACTOR, LR_PATIENCE, LR_MIN),   # NEW, not in the paper
    extra_model_options=tuple(MODEL_OPTIONS.items()), test_every=TEST_EVERY,      # NEW: periodic test scores
    upload_checkpoints=True, wandb_entity=WANDB_ENTITY,
)
print('Run:', config.run_name, '| model channels:', config.input_channels)
command = [sys.executable, '-u', '-m', 'firepred.cli.train', *argv_for(config)]
# Save the command, settings and versions beside the checkpoints (no secrets or environment values).
write_run_record(OUTPUT_DIR, config, code={
    'repo': REPO_URL, 'branch': REPO_BRANCH, 'commit': CODE_COMMIT, 'window_data': work.window_repo,
    'tessera_inputs': USE_TESSERA and TESSERA_INPUTS, 'torchvision': TORCHVISION_VERSION,
    'torch_xla': TORCH_XLA_VERSION, 'accelerator': ACCELERATOR,
})
subprocess.run(command, cwd=CODE_DIR, check=True)

print('Saved checkpoints:')
for checkpoint in sorted(OUTPUT_DIR.rglob('*.pth')):
    print(checkpoint)
print('Output directory:', OUTPUT_DIR)


## Notes and references
- `WANDB` and `HF_TOKEN` are secret **names** in the Colab Secrets panel, not values to paste into this notebook.
- The VM disk (`/content`) is wiped when the runtime disconnects; use `SAVE_TO_DRIVE = True` or the W&B
  model Artifacts to keep checkpoints.
- Keep `use_checkpoint=False`: the custom Swin checkpoint branch needs a separate fix.
- Training tests its own best checkpoint at the end. A separate `-test` run is only needed for an
  older checkpoint: pass `--checkpoint` with its path, `--tessera` exactly when it was trained with
  TESSERA inputs, and `--swin-temporal-downsample` when it was trained before the paper's
  no-time-merging change (the trainer refuses a mismatch).
- A successful smoke test verifies one step, not an entire 100-epoch run.

References: [Colab secrets (`google.colab.userdata`)](https://colab.research.google.com/),
[Mounting Google Drive in Colab](https://colab.research.google.com/notebooks/io.ipynb),
[Hugging Face downloads](https://huggingface.co/docs/huggingface_hub/guides/download),
[W&B model Artifacts and upload completion](https://docs.wandb.ai/models/ref/python/experiments/artifact).
